# Lab 19: The 10-Like Challenge

**Question:** Can a new Instagram post reach 10 likes in 10 minutes?

**Your job:** A creator wants to know how likely a new post is to reach **10 likes in 10 minutes**. You will build thousands of imaginary posts, compare the results with a Gamma probability model, and recommend how quickly likes need to arrive for the creator to have at least an **80% chance of success**.

**Assumption:** Likes arrive independently, at a constant average rate. Real social media activity may not follow this model exactly.

**Directions:** Run cells **top to bottom**. Complete lines marked `TODO`. For short explanations, type between the triple quotation marks (`"""`) in the answer cells—you don't need to know Markdown. Round numerical answers to **3 decimals** unless otherwise noted.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import math

rng = np.random.default_rng(105)
print("Ready to investigate!")

## Part 1: One post, ten likes

Your notes showed that if likes arrive at **1 like per minute on average**, each wait can be generated using

$$
\text{wait}=-\ln(U), \qquad 0<U<1.
$$

A post needs to wait for its **first like**, then its **second**, and so on. The time to the tenth like is the **sum of ten individual waiting times**.

In [ ]:
# Simulate the gaps between likes for ONE post.
U = rng.uniform(size=10)
waits = -np.log(U)                      # Each wait is exponential, rate = 1
arrival_times = np.cumsum(waits)       # Cumulative sum = time each like arrives

print("Like  |  Wait for this like  |  Total time elapsed")
print("-" * 50)
for i in range(10):
    print(f"{i+1:>4}  |       {waits[i]:>6.2f} min   |       {arrival_times[i]:>6.2f} min")
print(f"\nTime to 10th like: {arrival_times[-1]:.3f} minutes")

In [ ]:
answer_1 = """TODO: Does every post get 10 likes in exactly 10 minutes?
Explain how the table shows what 'time to the 10th like' means."""
print(answer_1)

## Part 2: What happens across 5,000 posts?

One post is just one random outcome. Simulate **5,000 different posts**, each waiting for ten likes.

The computer stores the time to each like in a table with **5,000 rows and 10 columns**. Python counts columns starting at **0** rather than 1.

**Your code task:** The first-like column is filled in as an example. Choose the correct columns for the third and tenth likes. Then run the graphs.

In [ ]:
N = 5000
U_many = rng.uniform(size=(N, 10))
waits_many = -np.log(U_many)
all_arrivals = np.cumsum(waits_many, axis=1)

T_first = all_arrivals[:, 0]
T_third = all_arrivals[:, 0]      # TODO: column for the THIRD like
T_tenth = all_arrivals[:, 0]      # TODO: column for the TENTH like
if np.array_equal(T_tenth, T_first):
    print("TODO CHECK: T_tenth is still using the first-like column.")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True, sharey=True)
for ax, times, label in zip(axes,
                            [T_first, T_third, T_tenth],
                            ["1st like", "3rd like", "10th like"]):
    ax.hist(times, bins=np.arange(0, 26, 1), density=True,
            edgecolor="white", alpha=0.8)
    ax.set_title(f"Time to {label}\nMean = {np.mean(times):.2f} min")
    ax.set_xlabel("Minutes since posting")
    ax.set_xlim(0, 25)
axes[0].set_ylabel("Density (area = probability)")
plt.tight_layout()
plt.show()

In [ ]:
answer_2 = """TODO: How do the peaks and average times change from the 1st to the
3rd to the 10th like?"""
print(answer_2)

## Part 3: Can we beat the 10-minute deadline?

The creator wants to reach **10 likes by 10 minutes**. Let's estimate the chance of success **two ways**.

**Method A: Data.** Count the proportion of your 5,000 simulated posts that succeeded.

**Method B: Integration.** For 1 like/minute, the waiting time to the tenth like has Gamma density

$$
f_{10}(t)=\frac{t^9e^{-t}}{9!}, \qquad t\geq 0.
$$

So the model's predicted probability is

$$
P(T_{10}\leq 10)=\int_0^{10}\frac{t^9e^{-t}}{9!}\,dt.
$$

The `quad` function computes the area under the curve numerically.

**Your code task:** Fix the two lines marked `TODO` to use the **10-minute** deadline.

In [ ]:
deadline = 10

# METHOD A: probability estimated from simulated posts
p_data = np.mean(T_tenth <= 1)    # TODO: use the correct time cutoff

# METHOD B: probability predicted by the Gamma model
def gamma10_density(t):
    return (t**9 * np.exp(-t)) / math.factorial(9)

p_model, _ = quad(gamma10_density, 0, 1)  # TODO: integral ends at deadline

print(f"Chance from simulation: {p_data:.3f}")
print(f"Chance from integration: {p_model:.3f}")
print(f"Difference:             {abs(p_data-p_model):.3f}")
if p_model < 0.01:
    print("TODO CHECK: The integral still uses the placeholder upper bound.")

In [ ]:
answer_3 = """TODO: What is the probability of success?"""
print(answer_3)

In [ ]:
t = np.linspace(0, 24, 500)
plt.figure(figsize=(8, 4))
plt.hist(T_tenth, bins=np.arange(0, 26, 1), density=True,
         alpha=0.35, edgecolor="white", label="Simulated posts")
plt.plot(t, gamma10_density(t), linewidth=2.5, label="Gamma density")
plt.fill_between(t, gamma10_density(t), where=(t <= deadline),
                 alpha=0.25,color = 'm', label="Probability of reaching 10 by 10 min")
plt.axvline(deadline, linestyle="--", label="10-minute deadline")
plt.xlabel("Minutes until the 10th like")
plt.ylabel("Probability density")
plt.legend()
plt.title("One probability, two approaches")
plt.tight_layout()
plt.show()

In [ ]:
answer_4 = """TODO: Explain what the purple shaded area represents. Why should the
simulation and integration be close but not exactly identical?"""
print(answer_4)

### Part 4: The creator's 80% challenge

The creator says: **"I want at least an 80% chance of getting ten likes by ten minutes. How much faster do likes need to arrive?"**

Let $\lambda$ be the **average number of likes per minute**. Until now $\lambda=1$.

For a different rate, the waiting-time density becomes

$$
f_{10,\lambda}(t)=\frac{\lambda^{10}t^9e^{-\lambda t}}{9!},\qquad t\geq 0.
$$

**Your code task:** Test rates in the table. Then **add some rates between 1.25 and 1.30** to find the smallest rate (to two decimal places) that reaches the goal.

In [ ]:
def gamma10_density_rate(t, rate):
    return (rate**10) * (t**9) * np.exp(-rate*t) / math.factorial(9)

def chance_by_deadline(rate, deadline=10):
    area, _ = quad(lambda t: gamma10_density_rate(t, rate),
                   0, deadline)
    return area

goal = 0.80
rate_to_try = 1  # TODO: add rates between 1.25 and 1.30

print("Likes/min   P(10 likes by 10 min)   Meets 80% goal?")
print("-" * 49)

prob = chance_by_deadline(rate_to_try)
success = "YES" if prob >= goal else "no"
print(f"{rate:>7.3f}          {prob:>7.3f}              {success}")

In [ ]:
t = np.linspace(0, 22, 500)
plt.figure(figsize=(8, 4))
for rate in [1.0, 1.3]:
    plt.plot(t, gamma10_density_rate(t, rate), linewidth=2.5,
             label=f"{rate} likes/min")
plt.axvline(10, linestyle="--", label="Deadline")
plt.xlabel("Minutes until the 10th like")
plt.ylabel("Probability density")
plt.title("What changes when likes arrive faster?")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
answer_5 = """TODO: Recommend the smallest rate (to 0.01 likes/min) that gives
at least an 80% chance. How many seconds between likes is that on average?
Does 80% mean every post succeeds? Name one way the real world could differ
from this mathematical model."""
print(answer_5)

### Final reflection

You used a random sample and an integral to answer the same question. Complete the final answer below, then submit your `.ipynb` file.

In [ ]:
final_answer = """TODO: Write a 2-3 sentence recommendation using BOTH a
probability and a mathematical explanation for why posts still vary."""
print(final_answer)

### Optional bonus: A calculus surprise

The average time to the tenth like is **10 minutes**, but the Gamma density peaks at a **different time**. Find where
$$
f_{10}(t)=\frac{t^9e^{-t}}{9!}
$$

has its maximum using a derivative. Why is this peak not the same as the average?

In [ ]:
bonus_answer = """OPTIONAL: Show the derivative and explain the difference
between the peak of the density and the average."""
print(bonus_answer)